In [1]:
!pip install sklearn-crfsuite stanza

In [ ]:
import random
import numpy as np
import stanza

from sklearn.model_selection import train_test_split
from sklearn_crfsuite import CRF
from sklearn_crfsuite import metrics

In [ ]:
stanza.download('ta')

nlp = stanza.Pipeline(
    lang='ta',
    processors='tokenize,pos',
    tokenize_no_ssplit=True,
    use_gpu=False
)

In [4]:
def load_data(filepath):

    sentences = []

    with open(filepath, encoding="utf8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            pairs = []

            i = 0
            while i < len(parts)-1:

                word = parts[i]
                label = parts[i+1]

                if label in ["NSIL","LSIL","MSIL","SSIL"]:
                    pairs.append((word,label))
                    i += 2
                else:
                    i += 1

            if pairs:
                sentences.append(pairs)

    return sentences


data = load_data("final_sentences_tamil_with_silence.txt")

print("Total sentences:",len(data))

Total sentences: 6365


In [5]:
def oversample_data(data):

    new_data = []

    for sent in data:

        labels = {l for _,l in sent}

        if "SSIL" in labels:
            new_data.extend([list(sent) for _ in range(6)])

        elif "MSIL" in labels:
            new_data.extend([list(sent) for _ in range(3)])

        else:
            new_data.append(sent)

    return new_data


data = oversample_data(data)

print("After oversampling:",len(data))

After oversampling: 24408


In [6]:
def get_pos_tags(words):

    doc = nlp(" ".join(words))

    pos_tags = []

    for sentence in doc.sentences:
        for word in sentence.words:
            pos_tags.append(word.upos)

    return pos_tags

In [7]:
def word2features(sent,pos_tags,i):

    word = sent[i][0]
    pos = pos_tags[i]

    features = {

        'bias':1.0,
        'word.lower':word.lower(),
        'word_length':len(word),
        'pos':pos,

        'prefix1':word[:1],
        'prefix2':word[:2],
        'prefix3':word[:3],

        'suffix1':word[-1:],
        'suffix2':word[-2:],
        'suffix3':word[-3:],

        'is_start': i==0,
        'is_end': i==len(sent)-1
    }

    if i>0:

        prev_word = sent[i-1][0]
        prev_pos = pos_tags[i-1]

        features.update({
            'prev_word':prev_word,
            'prev_pos':prev_pos
        })

    if i<len(sent)-1:

        next_word = sent[i+1][0]
        next_pos = pos_tags[i+1]

        features.update({
            'next_word':next_word,
            'next_pos':next_pos
        })

    return features

In [8]:
def sent2features(sent):

    words = [w for w,l in sent]

    pos_tags = get_pos_tags(words)

    return [word2features(sent,pos_tags,i) for i in range(len(sent))]

In [9]:
def sent2labels(sent):

    return [label for token, label in sent]

In [10]:
X = [sent2features(s) for s in data]

y = [sent2labels(s) for s in data]

In [11]:
X_train,X_test,y_train,y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42
)

In [12]:
crf = CRF(

    algorithm='lbfgs',

    c1=0.1,

    c2=0.1,

    max_iterations=500,

    all_possible_transitions=True
)

In [13]:
crf.fit(X_train,y_train)

,algorithm,'lbfgs'
,min_freq,None
,all_possible_states,None
,all_possible_transitions,True
,c1,0.1
,c2,0.1
,max_iterations,500
,num_memories,None
,epsilon,None
,period,None
,delta,None


In [14]:
y_pred = crf.predict(X_test)

In [15]:
labels = list(crf.classes_)

print(metrics.flat_classification_report(

    y_test,
    y_pred,
    labels=labels,
    digits=3
))

              precision    recall  f1-score   support

        MSIL      0.961     0.956     0.959     10719
        NSIL      0.960     0.967     0.964     19994
        LSIL      0.962     0.959     0.961     16096
        SSIL      0.973     0.961     0.967      2935

    accuracy                          0.962     49744
   macro avg      0.964     0.961     0.963     49744
weighted avg      0.962     0.962     0.962     49744



In [48]:
idx = random.randint(0, len(X_test)-1)

print("\nSample Prediction\n")

print(f"{'Word':<20} {'True_Label':<12} {'Predicted_Label':<15}")
print("-"*50)

for word, true, pred in zip(
    [w for w,l in data[idx]],
    y_test[idx],
    y_pred[idx]
):
    print(f"{word:<20} {true:<12} {pred:<15}")


Sample Prediction

Word                 True_Label   Predicted_Label
--------------------------------------------------
ஆனால்                MSIL         LSIL           
மிகப்                SSIL         SSIL           
பெரிய                MSIL         MSIL           
அளவில்               LSIL         LSIL           
நடத்தப்பட்ட          NSIL         NSIL           
அந்த                 MSIL         MSIL           
போராட்டத்தில்        NSIL         NSIL           
நாஞ்சில              NSIL         LSIL           
சம்பத்               LSIL         NSIL           


In [17]:
from sklearn_crfsuite import metrics

# predictions on training data
train_pred = crf.predict(X_train)

train_accuracy = metrics.flat_accuracy_score(y_train, train_pred)

print("Training Accuracy:", round(train_accuracy*100,2), "%")

Training Accuracy: 98.52 %


In [18]:
# predictions on test data
test_pred = crf.predict(X_test)

test_accuracy = metrics.flat_accuracy_score(y_test, test_pred)

print("Testing Accuracy:", round(test_accuracy*100,2), "%")

Testing Accuracy: 96.19 %


In [20]:
def sentence_to_features(sentence):

    words = sentence.strip().split()

    # POS tagging
    doc = nlp(" ".join(words))

    pos_tags = []

    for sent in doc.sentences:
        for word in sent.words:
            pos_tags.append(word.upos)

    sent_format = [(w,"") for w in words]

    features = [word2features(sent_format,pos_tags,i) for i in range(len(words))]

    return words, features

def predict_sentence(sentence):

    words, features = sentence_to_features(sentence)

    prediction = crf.predict([features])[0]

    print("\nPrediction Result\n")

    print(f"{'Word':<20} {'Predicted_Label':<15}")
    print("-"*40)

    for w,p in zip(words,prediction):
        print(f"{w:<20} {p:<15}")

user_sentence = input("Enter a Tamil sentence: ")

predict_sentence(user_sentence)

Enter a Tamil sentence:  ஓரியூர் சேதுபதியின் ஆட்சிக்கு அடங்கிய சிறு தலைக்கட்டு ஒருவரால் நிர்வாகம் செய்யப்பட்டு வந்தது



Prediction Result

Word                 Predicted_Label
----------------------------------------
ஓரியூர்              NSIL           
சேதுபதியின்          NSIL           
ஆட்சிக்கு            NSIL           
அடங்கிய              NSIL           
சிறு                 NSIL           
தலைக்கட்டு           LSIL           
ஒருவரால்             LSIL           
நிர்வாகம்            NSIL           
செய்யப்பட்டு         NSIL           
வந்தது               LSIL           
